#**Ajuste Fino de Hiperparâmetros**

Estudos e testes a respoeito do ajuste de hiperparametros de uma rede neural MLP. No exemplo, é usado o dataset de dígitos escritos a mão (MNIST)

## Configurações Prévias (imports, defs, etc.)

In [1]:
# if "google.colab" in sys.modules:
#     %pip install -q -U keras_tuner~=1.4.6

!pip install keras-tuner~=1.4.6

import sys
import sklearn
import numpy as np
import tensorflow as tf
import keras_tuner as kt

from tensorflow import keras
from tensorflow.keras import layers
from packaging import version

assert sys.version_info >= (3, 7)
assert version.parse(sklearn.__version__) >= version.parse("1.0.1")
assert version.parse(tf.__version__) >= version.parse("2.8.0")


In [2]:
tf.keras.backend.clear_session()
tf.random.set_seed(42)

##Modelo base

In [3]:
def build_model(hp):
    model_type  = hp.Choice("model_type", ["cnn"])
    num_filters = hp.Int("num_filters", 32, 128, step=32)
    dense_units = hp.Int("dense_units", 64, 256, step=64)

    # MNIST -> (28, 28, 1)
    inputs = keras.Input(shape=(28, 28, 1))
    x = inputs

    for i in range(hp.Int("conv_layers", 1, 3)):
        x = layers.Conv2D(
            filters=hp.Int(f"filters_{i}", 32, 128, step=32),
            kernel_size=3,
            activation="relu",
            padding="same"
        )(x)
        x = layers.MaxPooling2D()(x)

    x = layers.Flatten()(x)

    for _ in range(hp.Int("n_dense_layers", 1, 2)):
        x = layers.Dense(dense_units, activation="relu")(x)

    outputs = layers.Dense(10, activation="softmax")(x)

    model = keras.Model(inputs, outputs)

    model.compile(
        optimizer=keras.optimizers.Adam(
            hp.Float("lr", 1e-3, 1e-2, sampling="log")
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

##Carregamento e preparo dos dados (MNIST)

In [4]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalização (0–255 -> 0–1)
x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32") / 255.0

# Adicionar canal (28,28) -> (28,28,1)
x_train = np.expand_dims(x_train, axis=-1)
x_test  = np.expand_dims(x_test, axis=-1)

# Labels já estão no formato (N,)
print("x_train shape:", x_train.shape)
print("x_test shape:",  x_test.shape)
print("Numero de classes:", len(np.unique(y_train)),"\n")

val_size = 5000

x_val, x_train = x_train[:val_size], x_train[val_size:]
y_val, y_train = y_train[:val_size], y_train[val_size:]

x_train shape: (60000, 28, 28, 1)
x_test shape: (10000, 28, 28, 1)
Numero de classes: 10 



In [5]:
# val_size = np.random.permutation(len(x_train))
val_size = 5000

x_val, x_train = x_train[:val_size], x_train[val_size:]
y_val, y_train = y_train[:val_size], y_train[val_size:]

print("x_train:", x_train.shape)  # (45000, 32, 32, 3)
print("x_val:", x_val.shape)      # (5000, 32, 32, 3)

x_train: (50000, 28, 28, 1)
x_val: (5000, 28, 28, 1)


##RandomSearch

In [6]:
num_models = 5

tuner = kt.RandomSearch(
    build_model,
    objective="val_accuracy",
    max_trials=num_models,  # número de modelos testados
    executions_per_trial=3, # repetir treino (robustez)
    directory="tuner_dir",
    project_name="cifar10_search"
)

Reloading Tuner from tuner_dir/cifar10_search/tuner0.json


In [7]:
tuner.search(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=5,
    batch_size=64
)

In [8]:

trials = tuner.oracle.trials

results = []

for trial_id, trial in trials.items():
    hp = trial.hyperparameters.values
    score = trial.score  # val_accuracy

    row = hp.copy()
    row["val_accuracy"] = score

    results.append(row)

all_keys = set()
for r in results:
    all_keys.update(r.keys())

all_keys = sorted(all_keys)

header = " | ".join(f"{k:15s}" for k in all_keys)
print(header)
print("-" * len(header))

# linhas
for r in results:
    row = []
    for k in all_keys:
        val = r.get(k, "-")

        if isinstance(val, float):
            if "accuracy" in k:
                val = f"{val*100:.2f}%"   # porcentagem
            else:
                val = f"{val:.4f}"

        row.append(f"{str(val):15s}")

    print(" | ".join(row))

conv_layers     | dense_units     | filters_0       | filters_1       | filters_2       | lr              | model_type      | n_dense_layers  | num_filters     | val_accuracy   
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
2               | 128             | 32              | 32              | -               | 0.0015          | cnn             | 2               | 96              | 98.67%         
3               | 128             | 96              | 32              | 32              | 0.0073          | cnn             | 1               | 96              | 98.25%         
1               | 192             | 128             | -               | -               | 0.0094          | cnn             | 1               | 32              | 97.94%         
1               | 128             | 64              | -               | -               | 0.0021          | cn

##Recuperar o melhor modelo

In [9]:
best_model = tuner.get_best_models(num_models=1)[0]
best_hp    = tuner.get_best_hyperparameters(1)[0]

def print_hp_table(hp):
    print("\n========================================")
    print("        RELATÓRIO DE HIPERPARÂMETROS")
    print("========================================")

    print(f"{'PARAMETRO':20s} | {'VALOR':15s}")
    print("-" * 40)

    for key, value in sorted(hp.values.items()):
        print(f"{key:20s} | {str(value):15s}")

    print("-" * 40)

print_hp_table(best_hp)


        RELATÓRIO DE HIPERPARÂMETROS
PARAMETRO            | VALOR          
----------------------------------------
conv_layers          | 2              
dense_units          | 128            
filters_0            | 32             
filters_1            | 32             
lr                   | 0.0015132200570019782
model_type           | cnn            
n_dense_layers       | 2              
num_filters          | 96             
----------------------------------------


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


##Treinar melhor modelo mais a fundo

In [10]:
from keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True, # Keeps the best performing weights
    verbose=1
)

history = best_model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=10,
    validation_split=0.2,
    callbacks=[early_stopping],
    batch_size=64
)

final_train_acc = history.history["accuracy"][-1]
final_val_acc   = history.history["val_accuracy"][-1]

print("\n==============================")
print(" RESULTADO FINAL")
print("==============================")

print(f"Acurácia final (treino): {final_train_acc * 100:.2f}%")
print(f"Acurácia final (val):    {final_val_acc * 100:.2f}%")

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 34s 40ms/step - accuracy: 0.9936 - loss: 0.0191 - val_accuracy: 0.9860 - val_loss: 0.0426
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 28s 36ms/step - accuracy: 0.9952 - loss: 0.0149 - val_accuracy: 0.9886 - val_loss: 0.0562
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 42s 37ms/step - accuracy: 0.9960 - loss: 0.0132 - val_accuracy: 0.9870 - val_loss: 0.0622
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 29s 37ms/step - accuracy: 0.9963 - loss: 0.0113 - val_accuracy: 0.9838 - val_loss: 0.0632
Epoch 4: early stopping
Restoring model weights from the end of the best epoch: 1.

 RESULTADO FINAL
Acurácia final (treino): 99.63%
Acurácia final (val):    98.38%
